In [92]:
import pandas as pd 
import numpy as np
from scipy import stats 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import PowerTransformer

# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import warnings

warnings.filterwarnings('ignore')
#sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#statistical libraries 
from scipy import stats
from scipy.stats import zscore, skew 

# set style for better visualizations 
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("libraries imported successfully!")


libraries imported successfully!


In [85]:
loan_df = pd.read_csv('home_loan_train.csv')

df = loan_df.copy()


In [86]:
df.dropna(subset=['Credit_History'],inplace=True)

In [87]:
df.isnull().sum()

Loan_ID               0
Gender               12
Married               3
Dependents           15
Education             0
Self_Employed        26
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           21
Loan_Amount_Term     14
Credit_History        0
Property_Area         0
Loan_Status           0
dtype: int64

In [88]:
numerical =df.select_dtypes(include=['float64','int64']).columns

numerical=numerical.drop('Credit_History')

categorical = df.select_dtypes(include='object').columns
categorical = list(categorical)
categorical.append('Credit_History')

In [89]:
for col in categorical:
 x=df[col].mode()[0]
 df[col]=df[col].fillna(x)

for col in numerical:
  df[col]=df[col].fillna(df[col].median())

In [90]:
df.duplicated().sum()

np.int64(0)

In [96]:
skewed_vars = ['LoanAmount', 'ApplicantIncome', 'CoapplicantIncome']

for var in skewed_vars:
    if var in df.columns:
        if df[var].skew() > 0:
            # Check if variable has zero or negative values
            min_val = df[var].min()
            if min_val <= 0:
                # Use log1p for variables with zeros
                df[f'{var}_log'] = np.log1p(df[var])
                print(f"✓ {var}: Applied log1p transformation (had {min_val:.3f} minimum value)")
            else:
                # Use log for positive values only
                df[f'{var}_log'] = np.log(df[var])
                print(f"✓ {var}: Applied log transformation")
        else:
            pt = PowerTransformer(method='yeo-johnson')
            df[var] = pt.fit_transform(df[var])
    # Check skewness before and after
    original_skew = skew(df[var])
    transformed_skew = skew(df[f'{var}_log'])
    print(f"  Original skewness: {original_skew:.3f} → Transformed skewness: {transformed_skew:.3f}")

print(f"\nDataset shape after log transformation: {df.shape}")
print("New log-transformed columns:", [col for col in df.columns if '_log' in col])


✓ LoanAmount: Applied log transformation
  Original skewness: 2.698 → Transformed skewness: -0.284
✓ ApplicantIncome: Applied log transformation
  Original skewness: 6.490 → Transformed skewness: 0.539
✓ CoapplicantIncome: Applied log1p transformation (had 0.000 minimum value)
  Original skewness: 5.985 → Transformed skewness: -0.153

Dataset shape after log transformation: (564, 16)
New log-transformed columns: ['LoanAmount_log', 'ApplicantIncome_log', 'CoapplicantIncome_log']


consider other transformation methods on subsequent evaluations because its over flipping the data i am now having negative skewness 